In [ ]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [ ]:
def run_sql_file(path):
    with open(path,'r') as f:
        script=f.read()
    con.executescript(script)
    con.commit()
    print('ran',path)

In [ ]:
run_sql_file('../sql/practice/01_revenue_customer.sql')

In [ ]:
q("""select 
  count(*) as total_customers,
  ROUND(AVG(arpu_inr),2) as average_recorder_arpu,
  COUNT(DISTINCT arpu_inr) as distinct_arpu_values
  from customers;""")

In [ ]:
q("""
SELECT 
    COUNT(*) AS customers_without_payments
FROM customers AS c
WHERE NOT EXISTS (
    SELECT 1 
    FROM recharges AS r
    WHERE r.customer_id = c.customer_id
      AND r.recharge_date >= '2025-01-01'
      AND r.recharge_date < '2025-07-01'
);
""")

In [ ]:
q(""" select name,type
from sqlite_master
where name ='v_revenue_customer';""")

In [ ]:
q("""select* from v_revenue_customer;""")

In [ ]:
run_sql_file('../sql/practice/02_usage_customer.sql')

In [ ]:
q("""
SELECT 
    SUM(CASE WHEN recharge_count > 0 THEN 1 ELSE 0 END) 
        AS customers_with_payments,
    SUM(CASE WHEN recharge_count = 0 THEN 1 ELSE 0 END) 
        AS customers_without_payments
FROM v_revenue_customer;
""")

In [ ]:
q("""select * from  v_usage_customer""")

In [ ]:
q("""select count(*) as customers_without_complaints
   from customers as c 
   left join complaints as cp 
     on  c.customer_id == cp.customer_id
 where cp.customer_id is null;""")

In [ ]:
run_sql_file('../sql/practice/03_complaints_customer.sql')

In [ ]:
q("""select count(*) from v_complaint_customer;""")

In [ ]:
q("""select * from v_complaint_customer;""")

In [ ]:
q(""" select count(*)
 from v_complaint_customer
 where complaint_count=0;""")

In [ ]:
run_sql_file('../sql/practice/04_plans_clean.sql')

In [ ]:
q(""" select * from v_plans_clean;""")

In [ ]:
run_sql_file('../sql/practice/05_usage_month.sql')

In [ ]:
q("""select * from v_usage_month;""")

In [ ]:
run_sql_file('../sql/practice/06_customer_features.sql')

In [ ]:
q("""select * from customer_features;""")

In [ ]:
q("""SELECT COUNT(*)                  AS rows_,
       COUNT(DISTINCT customer_id) AS subs,
       SUM(usage_days = 0)         AS no_usage,
       SUM(recharge_count = 0)     AS no_payment,
       SUM(complaint_count = 0)    AS never_complained,
       ROUND(AVG(tenure_days), 1)  AS tenure,
       ROUND(AVG(actual_arpu_6m), 2) AS actual_arpu,
       ROUND(100.0*AVG(is_churned), 2) AS churn
FROM   customer_features;""")

In [ ]:
q("""SELECT c.customer_id,                      -- one row per sub
         NTILE(4) OVER (ORDER BY             -- 1 = longest
           COALESCE(u.recency_days, 999)     -- silence
           DESC) AS r_score,                 -- the R score
         NTILE(4) OVER (ORDER BY             -- 4 = pays most
           r.recharge_count) AS f_score,     -- often
         NTILE(4) OVER (ORDER BY             -- 4 = pays the
           r.recharge_total_6m) AS m_score   -- most money
  FROM   customers c                         -- all 22,000
  JOIN   v_usage_customer   u ON u.customer_id = c.customer_id -- Fix 2
  JOIN   v_revenue_customer r ON r.customer_id = c.customer_id ;-- Fix 1;""")